# CHP 08 Extending PySpark with Python: RDD and UDFs

RDD: Resilient Distributed DataSet (underlying object implementing dataframes. Act as a collection of objects instead of a set of rows and columns)
 - can think of each row of a dataframe as an object in an RDD

UDF: User Defined Function

Objects in an RDD are typically modified through the following operations, map(), filter(), and reduce().
These three options are akin to the functional programming concepts map, filter, and reduce found in Java Streams and JavaScript arrays.
Each of these operations are "higher order" functions since they take in other functions as inputs.

- map() -> applies an input function to every element in a collection
- filter() -> picks out/in elements based on an input fiter criterion function
- reduce() -> consolidates a collection of elements into a single element based off of an input consolidation function and an optional default initial value (e.g. for single element lists)

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

sc = spark.sparkContext

your 131072x1 screen size is bogus. expect trouble
25/02/02 09:45:44 WARN Utils: Your hostname, LAPTOP-CDHH1LA0 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/02/02 09:45:44 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/02/02 09:45:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
collection = [1, "two", 3.0, ("four", 4), {"five":5}]
# create a resilient distributed dataset from the array by parallizing it via the spark context
collection_rdd = sc.parallelize(collection)

In [3]:
print(collection_rdd)

ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:289


In [4]:
# mapping

from py4j.protocol import Py4JJavaError

def add_one(value):
    """We expect this function to throw an error when it operates on a non-numeric type"""
    return value + 1;

collection_rdd_m = collection_rdd.map(add_one) # attempt to apply the "add_one" to every element in the RDD

try:
    print(collection_rdd_m.collect()) # collect() materializes an RDD into a python lis on the master node
except Py4JJavaError as e:
    pass



25/02/02 09:45:48 ERROR Executor: Exception in task 9.0 in stage 0.0 (TID 9) 12]
org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/home/hubert/spark-3.5.1-bin-hadoop3/python/lib/pyspark.zip/pyspark/worker.py", line 1247, in main
    process()
  File "/home/hubert/spark-3.5.1-bin-hadoop3/python/lib/pyspark.zip/pyspark/worker.py", line 1239, in process
    serializer.dump_stream(out_iter, outfile)
  File "/home/hubert/spark-3.5.1-bin-hadoop3/python/lib/pyspark.zip/pyspark/serializers.py", line 274, in dump_stream
    vs = list(itertools.islice(iterator, batch))
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/hubert/spark-3.5.1-bin-hadoop3/python/lib/pyspark.zip/pyspark/util.py", line 83, in wrapper
    return f(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_12558/3692626150.py", line 7, in add_one
TypeError: can only concatenate tuple (not "int") to tuple

	at org.apache.spark.api.python.BasePythonRunner$ReaderI

In [5]:
import logging

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

def safer_add_one(value):
    """same as add_one but safer. in case of type error it will return the original value while logging the error with stack trace"""
    try:
        return value + 1
    except TypeError as er:
        logger.warning(f"Error encountered: {er}", stack_info=True)
        return value

collection_rdd_m = collection_rdd.map(safer_add_one)

print(collection_rdd_m.collect())

[2, 'two', 4.0, ('four', 4), {'five': 5}]


Error encountered: unsupported operand type(s) for +: 'dict' and 'int'
Stack (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/hubert/spark-3.5.1-bin-hadoop3/python/lib/pyspark.zip/pyspark/daemon.py", line 218, in <module>
  File "/home/hubert/spark-3.5.1-bin-hadoop3/python/lib/pyspark.zip/pyspark/daemon.py", line 193, in manager
  File "/home/hubert/spark-3.5.1-bin-hadoop3/python/lib/pyspark.zip/pyspark/daemon.py", line 74, in worker
  File "/home/hubert/spark-3.5.1-bin-hadoop3/python/lib/pyspark.zip/pyspark/worker.py", line 1247, in main
    process()
  File "/home/hubert/spark-3.5.1-bin-hadoop3/python/lib/pyspark.zip/pyspark/worker.py", line 1239, in process
    serializer.dump_stream(out_iter, outfile)
  File "/home/hubert/spark-3.5.1-bin-hadoop3/python/lib/pyspark.zip/pyspark/serializers.py", line 274, in dump_stream
    vs = list(itertools.islice(iterator, batch))
  File "/home/hubert/sp

## Use filter to filter in stuff (kind of like where in sql inspired domains)

In [7]:
collection_rdd_m = collection_rdd.filter(lambda elem: isinstance(elem, (float, int)))

print(collection_rdd.collect())
print(collection_rdd_m.collect())

[1, 'two', 3.0, ('four', 4), {'five': 5}]
[1, 3.0]


## Use Reduce to take two items and combine them into one, similar to groupby/agg in sql land

Since reduce is distributed (applied independently on each partition), any function you use needs to be commutative and associative.

Commutative means that the order of arguments is not important. E.G. add is commutative but subtract is not.
An associative function means that the grouping of arguments is not important. Add is associative, but subtract is not.

In [13]:
from operator import add

aSum = (
    sc.parallelize([4,7,9,1,3]).reduce(add)
)
print(aSum)

print(sc.parallelize([1]).reduce(add))
#print(sc.parallelize([]).reduce(add)) # fails b/c apparently reduce can't handle empty rdds

24
1


In [15]:
### E8.1
# reimplement count with map, filter, and/or reduce
logger.info(f"{sc.parallelize([1,2,3]).count()}")
logger.info(f"{sc.parallelize([1,2,3]).map(lambda x: 1).reduce(lambda a,b: a+ b)}")

INFO:__main__:3
INFO:__main__:3


In [16]:
### E8.2
# what is the return value of the following code block?
a_rdd = sc.parallelize([0, 1, None, [], 0.0])
 
a_rdd.filter(lambda x: x).collect()

# returns [1] b/c everything else is falsey

[1]

### Spark dataframes are RDDs

In [ ]:
df = spark.createDataFrame([[1], [2], [3]], schema=["column"])

print(df.rdd)
print(df.rdd.collect()) # the output of this call reveals that a spark dataframe can be seen as a RDD of "Row" where each Row is like a dictionary of key value pairs

# there is a conversion cost to go from dataframe (column major) to rdd (row major) so don't do it too often
# you also lose schema safety associated with the dataframe

MapPartitionsRDD[57] at javaToPython at NativeMethodAccessorImpl.java:0
[Row(column=1), Row(column=2), Row(column=3)]


## Extending Pyspark with Python UDFs

In [19]:
import pyspark.sql.functions as F 
import pyspark.sql.types as T 

fractions = [[x,y] for x in range(100) for y in range(1, 100)]
frac_df = spark.createDataFrame(fractions, ["numerator", "denominator"])

frac_df = frac_df.select(
    F.array(F.col("numerator"), F.col("denominator")).alias("fraction")
)
frac_df.show(5, False)

+--------+
|fraction|
+--------+
|[0, 1]  |
|[0, 2]  |
|[0, 3]  |
|[0, 4]  |
|[0, 5]  |
+--------+
only showing top 5 rows



In [23]:
from fractions import Fraction
from typing import Tuple, Optional

Frac = Tuple[int, int]

def py_reduce_fraction(frac: Frac) -> Optional[Frac]:
    """Reduce a fraction represented as a 2 tuple of integers"""
    num, denom = frac 
    if denom:
        answer = Fraction(num, denom)
        return answer.numerator, answer.denominator
    return None 
assert py_reduce_fraction((3, 6)) == (1, 2)
assert py_reduce_fraction((1, 0)) is None

def py_fraction_to_float(frac: Frac) -> Optional[Frac]:
    """Transforms a fraction represented as a 2 tuple of integers into a float"""
    num, denom = frac 
    if denom:
        return num/denom 
    return None 

assert py_fraction_to_float((2,8)) == 0.25
assert py_fraction_to_float((3,0)) == None

### Promoting python functions to UDFs with udf
pyspark.sql.functions.udf() function takes two args:
    1. the function to promote
    2. the return type of the generated udf
If a return type is provided it must be compatible with the return type of the udf

In [24]:
SparkFrac = T.ArrayType(T.LongType()) # spark type that is compatible with the return type expected from py_reduce_fraction

reduce_fraction_udf = F.udf(py_reduce_fraction, SparkFrac)

frac_df2 = frac_df.withColumn("reduced_fraction", reduce_fraction_udf(F.col("fraction")))

frac_df2.show(5, False)

+--------+----------------+
|fraction|reduced_fraction|
+--------+----------------+
|[0, 1]  |[0, 1]          |
|[0, 2]  |[0, 1]          |
|[0, 3]  |[0, 1]          |
|[0, 4]  |[0, 1]          |
|[0, 5]  |[0, 1]          |
+--------+----------------+
only showing top 5 rows



In [31]:
@F.udf(T.DoubleType())
def fraction_to_float(frac: Frac) -> Optional[float]:
    num, denom = frac 
    if denom:
        return num / denom 
    return None 

frac_df3 = frac_df.withColumn("fraction_float", fraction_to_float(F.col("fraction")))

frac_df3.select("fraction", "fraction_float").distinct().show(5, False)
assert fraction_to_float.func((1,2)) == 0.5 # you can access the underlying function with the func method.

+--------+--------------------+
|fraction|fraction_float      |
+--------+--------------------+
|[2, 4]  |0.5                 |
|[3, 50] |0.06                |
|[3, 67] |0.04477611940298507 |
|[7, 76] |0.09210526315789473 |
|[3, 76] |0.039473684210526314|
+--------+--------------------+
only showing top 5 rows



In [ ]:
conv2c = {
    "c": lambda x: x,
    "k": lambda x: x - 273.15,
    "r": lambda x: (x - 491.67) * (5/9)
}

convFrmC = {
    "c": lambda x: x,
    "k": lambda x: x + 273.15,
    "r": lambda x: 491.67 + (9/5) * x
}

def temp_to_temp(value: float, frm: str, to: str) -> Optional[float]:
    temp_as_c = conv2c[frm](value)
    return convFrmC[to](temp_as_c)


In [ ]:
### Ex 8.4

@F.udf((T.FloatType()))
def naive_udf(t: float) -> Optional[float]:
    return t * 3.14159